# Issue #4: Raw Data Preprocessing for Modeling

This notebook builds the modeling dataset from **raw 15-minute SVC volume data (2015-2019)** to align with faculty requirements. Summary datasets are used only for optional cross-checking, not for target creation.

## 1) Setup and Path Resolution

We detect the repository root and load files using robust candidate paths so the notebook works both in-repo and with `/mnt/data` fallbacks.

In [1]:
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate
    return start

repo_root = find_repo_root(Path.cwd())
print(f"Repo root: {repo_root}")

raw_candidates = [
    repo_root / "data" / "raw" / "svc_raw_data_volume_2015_2019.csv",
    Path("/mnt/data/svc_raw_data_volume_2015_2019.csv"),
]

summary_candidates = [
    repo_root / "data" / "raw" / "svc_summary_data.csv",
    Path("/mnt/data/svc_summary_data.csv"),
]

raw_path_opt: Optional[Path] = next((p for p in raw_candidates if p.exists()), None)
if raw_path_opt is None:
    raise FileNotFoundError(f"Could not find raw file in candidates: {raw_candidates}")
raw_path: Path = raw_path_opt

summary_path: Optional[Path] = next((p for p in summary_candidates if p.exists()), None)

processed_dir = repo_root / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)
output_path = processed_dir / "modeling_dataset_2015_2019.csv"

print(f"Using raw input: {raw_path}")
print(f"Summary cross-check input: {summary_path}")
print(f"Output path: {output_path}")


Repo root: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting
Using raw input: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/raw/svc_raw_data_volume_2015_2019.csv
Summary cross-check input: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/raw/svc_summary_data.csv
Output path: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/processed/modeling_dataset_2015_2019.csv


## 2) Load and Clean Raw 15-Minute Data

Cleaning rules:
- Parse datetime from `time_start`
- Convert `volume_15min` to numeric
- Drop rows with invalid/missing datetime or volume
- Remove duplicate records
- Restrict to years 2015-2019 as required for Issue #4

In [2]:
assert raw_path is not None
raw_df = pd.read_csv(raw_path)
print(f"Raw shape before cleaning: {raw_df.shape}")
print("Columns:", raw_df.columns.tolist())

# Build a stable location identifier from centreline_id + direction when available.
if "centreline_id" in raw_df.columns:
    centreline_source = raw_df["centreline_id"]
else:
    centreline_source = pd.Series(np.nan, index=raw_df.index)

centreline = pd.to_numeric(centreline_source, errors="coerce").astype("Int64")
base_loc = centreline.astype(str)
missing_base = centreline.isna()

if "location_name" in raw_df.columns:
    location_name_series = raw_df["location_name"].astype(str).str.strip()
else:
    location_name_series = pd.Series("UNKNOWN", index=raw_df.index, dtype="string")

base_loc = base_loc.mask(missing_base, location_name_series)

if "direction" in raw_df.columns:
    direction = raw_df["direction"].astype(str).str.strip()
else:
    direction = pd.Series("UNK", index=raw_df.index, dtype="string")

raw_df["location_id"] = base_loc + "_" + direction.replace({"": "UNK"})

raw_df["time_start"] = pd.to_datetime(raw_df["time_start"], errors="coerce")
raw_df["volume_15min"] = pd.to_numeric(raw_df["volume_15min"], errors="coerce")

before_drop = len(raw_df)
raw_df = raw_df.dropna(subset=["time_start", "volume_15min"]).copy()
after_drop = len(raw_df)

# Enforce required training years.
raw_df = raw_df[raw_df["time_start"].dt.year.between(2015, 2019)].copy()

# Remove exact duplicate observations at location/time/volume granularity.
before_dupes = len(raw_df)
raw_df = raw_df.drop_duplicates(subset=["location_id", "time_start", "volume_15min"])
after_dupes = len(raw_df)

print(f"Dropped invalid datetime/volume rows: {before_drop - after_drop:,}")
print(f"Removed duplicate 15-min observations: {before_dupes - after_dupes:,}")
print(f"Cleaned raw shape: {raw_df.shape}")
print(f"Year range: {raw_df['time_start'].dt.year.min()}-{raw_df['time_start'].dt.year.max()}")


Raw shape before cleaning: (556608, 10)
Columns: ['id', 'count_id', 'location_name', 'longitude', 'latitude', 'centreline_id', 'time_start', 'time_end', 'direction', 'volume_15min']


Dropped invalid datetime/volume rows: 0
Removed duplicate 15-min observations: 199
Cleaned raw shape: (556409, 11)
Year range: 2015-2019


## 3) Aggregate 15-Minute to Hourly, then Daily

Aggregation logic:
- **Hourly**: sum all 15-minute records per `location_id` and hour
- **Daily**: sum hourly totals per `location_id` and date
- **Peak hour volume**: maximum hourly volume per location-date
- **Peak ratio**: `peak_hour_volume / daily_total_volume`

`congestion_target` is defined as raw-derived `daily_total_volume`.

In [3]:
raw_df["hour"] = raw_df["time_start"].dt.floor("h")

hourly = (
    raw_df.groupby(["location_id", "hour"], as_index=False)
    .agg(
        hourly_volume=("volume_15min", "sum"),
        location_name=("location_name", "first"),
        centreline_id=("centreline_id", "first"),
    )
)

hourly["date"] = hourly["hour"].dt.floor("D")

daily_total = (
    hourly.groupby(["location_id", "date"], as_index=False)
    .agg(
        daily_total_volume=("hourly_volume", "sum"),
        location_name=("location_name", "first"),
        centreline_id=("centreline_id", "first"),
    )
)

peak_hour = (
    hourly.groupby(["location_id", "date"], as_index=False)
    .agg(peak_hour_volume=("hourly_volume", "max"))
)

modeling_df = daily_total.merge(peak_hour, on=["location_id", "date"], how="left")
modeling_df["peak_ratio"] = np.where(
    modeling_df["daily_total_volume"] > 0,
    modeling_df["peak_hour_volume"] / modeling_df["daily_total_volume"],
    np.nan,
)

modeling_df["year"] = modeling_df["date"].dt.year
modeling_df["month"] = modeling_df["date"].dt.month
modeling_df["day_of_week"] = modeling_df["date"].dt.dayofweek
modeling_df["is_weekend"] = modeling_df["day_of_week"].isin([5, 6]).astype(int)
modeling_df["congestion_target"] = modeling_df["daily_total_volume"]

ordered_cols = [
    "location_id",
    "location_name",
    "centreline_id",
    "date",
    "year",
    "month",
    "day_of_week",
    "is_weekend",
    "daily_total_volume",
    "peak_hour_volume",
    "peak_ratio",
    "congestion_target",
]
modeling_df = modeling_df[ordered_cols].sort_values(["location_id", "date"]).reset_index(drop=True)

print(f"Hourly shape: {hourly.shape}")
print(f"Modeling daily shape: {modeling_df.shape}")
modeling_df.head()


Hourly shape: (138672, 6)
Modeling daily shape: (5778, 12)


,location_id,location_name,centreline_id,date,year,month,day_of_week,is_weekend,daily_total_volume,peak_hour_volume,peak_ratio,congestion_target
0,10010625_WB,Danforth Ave: Donlands Ave - Byron Ave,10010625,2015-05-14,2015,5,3,0,17031,1809,0.106218,17031
1,10010625_WB,Danforth Ave: Donlands Ave - Byron Ave,10010625,2015-05-15,2015,5,4,0,17365,1734,0.099856,17365
2,10010625_WB,Danforth Ave: Donlands Ave - Byron Ave,10010625,2015-05-16,2015,5,5,1,14470,1066,0.073670,14470
3,10010625_WB,Danforth Ave: Donlands Ave - Byron Ave,10010625,2015-05-17,2015,5,6,1,12235,973,0.079526,12235
4,10010625_WB,Danforth Ave: Donlands Ave - Byron Ave,10010625,2015-05-18,2015,5,0,0,16523,1775,0.107426,16523


## 4) Persist Processed Modeling Dataset

This CSV is the canonical input for modeling and EDA notebooks in Issue #4.

In [4]:
modeling_df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")
print(f"Rows: {len(modeling_df):,}")
print(f"Columns: {modeling_df.columns.tolist()}")


Saved: /Users/manavparikh/Documents/GitHub/unfc-capstone-traffic-forecasting/data/processed/modeling_dataset_2015_2019.csv
Rows: 5,778
Columns: ['location_id', 'location_name', 'centreline_id', 'date', 'year', 'month', 'day_of_week', 'is_weekend', 'daily_total_volume', 'peak_hour_volume', 'peak_ratio', 'congestion_target']


## 5) Optional Validation Against Summary Data (Cross-Check Only)

This check validates aggregation consistency on a small sample.
It does **not** define modeling targets from summary data.

In [5]:
if summary_path is None:
    print("Summary file not found; skipping optional cross-check.")
else:
    summary_df = pd.read_csv(summary_path, usecols=["count_id", "avg_daily_vol"])

    count_daily = (
        raw_df.assign(date=raw_df["time_start"].dt.floor("D"))
        .groupby(["count_id", "date"], as_index=False)
        .agg(raw_daily_total=("volume_15min", "sum"))
    )

    raw_count_avg = (
        count_daily.groupby("count_id", as_index=False)
        .agg(raw_avg_daily_vol=("raw_daily_total", "mean"))
    )

    merged_check = raw_count_avg.merge(summary_df.drop_duplicates(subset=["count_id"]), on="count_id", how="inner")
    merged_check["abs_diff"] = (merged_check["raw_avg_daily_vol"] - merged_check["avg_daily_vol"]).abs()
    merged_check["pct_diff"] = np.where(
        merged_check["avg_daily_vol"].abs() > 0,
        merged_check["abs_diff"] / merged_check["avg_daily_vol"].abs(),
        np.nan,
    )

    print(f"Cross-check matched count_ids: {len(merged_check):,}")
    if len(merged_check) > 0:
        print("Mean absolute diff:", round(merged_check["abs_diff"].mean(), 3))
        print("Median absolute diff:", round(merged_check["abs_diff"].median(), 3))
        print("Median pct diff:", round(merged_check["pct_diff"].median(), 4))
        display_cols = ["count_id", "raw_avg_daily_vol", "avg_daily_vol", "abs_diff", "pct_diff"]
        merged_check.sort_values("abs_diff", ascending=False).head(10)[display_cols]


Cross-check matched count_ids: 1,026
Mean absolute diff: 0.205
Median absolute diff: 0.029
Median pct diff: 0.0


## Outputs

- `data/processed/modeling_dataset_2015_2019.csv`